# IAM Handwritten Text Recognition — Colab

This workflow clones project code from GitHub into temporary `/content`, reads the dataset archive from Google Drive, extracts IAM locally in `/content`, and stores only training outputs persistently in Drive. It reuses the repository configuration and training implementation; model architecture and hyperparameters are unchanged.

Before running: select a GPU runtime; set `GITHUB_REPO_URL` in the first code cell; upload `iam_words_colab.tar.gz` to `My Drive/handwriting_data/` (or edit `DRIVE_ARCHIVE_PATH`). The notebook does not download the dataset or start full training automatically. It runs a small GPU smoke test before the separate manual full-training cell.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# TODO: replace with your GitHub repository clone URL.
GITHUB_REPO_URL = 'https://github.com/YOUR_USERNAME/handwritten_text_recognition.git'

# Upload iam_words_colab.tar.gz to this Drive path, or change this setting.
DRIVE_ARCHIVE_PATH = Path('/content/drive/MyDrive/handwriting_data/iam_words_colab.tar.gz')
PROJECT_DIR = Path('/content/handwritten_text_recognition')
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/handwriting_training/handwritten_text_recognition')
SMOKE_TEST_PASSED = False
print('Repository URL setting:', GITHUB_REPO_URL)
print('Dataset archive:', DRIVE_ARCHIVE_PATH)
print('Persistent training outputs:', DRIVE_OUTPUT_DIR)

## Clone project code to Colab local storage

The repository must be public or otherwise accessible to this Colab runtime. For a private repository, configure GitHub authentication securely in Colab Secrets; do not paste a token into this notebook.

In [ ]:
import os
import subprocess
import sys

if 'YOUR_USERNAME' in GITHUB_REPO_URL or not GITHUB_REPO_URL.startswith(('https://', 'git@')):
    raise ValueError('Set GITHUB_REPO_URL to your actual GitHub repository clone URL.')
if PROJECT_DIR.exists():
    if not (PROJECT_DIR / '.git').is_dir():
        raise FileExistsError(f'{PROJECT_DIR} exists and is not a Git checkout; refusing to overwrite it.')
    remote = subprocess.check_output(
        ['git', '-C', str(PROJECT_DIR), 'remote', 'get-url', 'origin'], text=True
    ).strip().removesuffix('.git')
    if remote != GITHUB_REPO_URL.removesuffix('.git'):
        raise ValueError(f'Existing checkout remote {remote!r} differs from GITHUB_REPO_URL.')
else:
    subprocess.check_call(['git', 'clone', '--depth', '1', GITHUB_REPO_URL, str(PROJECT_DIR)])

for path in ('configs/config.yaml', 'requirements.txt', 'src/train.py', 'src/dataset.py'):
    if not (PROJECT_DIR / path).is_file():
        raise FileNotFoundError(f'Cloned repository is missing {path}')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Code checkout:', PROJECT_DIR)

## Install project dependencies and verify GPU

This reads `requirements.txt`, leaves the existing Colab PyTorch installation untouched, installs only missing project runtime packages, then checks imports and CUDA.

In [ ]:
import importlib.util
import subprocess
import sys

requirements = [
    line.strip().split('#', 1)[0].strip()
    for line in (PROJECT_DIR / 'requirements.txt').read_text(encoding='utf-8').splitlines()
]
requirements = [line for line in requirements if line]
module_for_package = {'numpy': 'numpy', 'pillow': 'PIL', 'pyyaml': 'yaml'}
to_install = []
for requirement in requirements:
    package = requirement.split('>=')[0].split('==')[0].split('~=')[0].strip().lower()
    if package == 'torch' or package == 'pytest':
        continue
    module = module_for_package.get(package)
    if module is None:
        raise RuntimeError(f'Unknown project dependency in requirements.txt: {requirement}')
    if importlib.util.find_spec(module) is None:
        to_install.append(requirement)
if to_install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *to_install])

import numpy
import PIL
import torch
import yaml

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select Runtime → Change runtime type → GPU, then rerun this cell.')
print('PyTorch:', torch.__version__, '(existing Colab installation; not replaced)')
print('NumPy:', numpy.__version__, '| Pillow:', PIL.__version__, '| PyYAML:', yaml.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))

## Inspect and extract the Drive archive

Default Drive path: `/content/drive/MyDrive/handwriting_data/iam_words_colab.tar.gz`. The archive is never modified. Its internal nesting is detected by finding the unique directory containing `words.txt` plus PNG files beneath its sibling `words/`. Extraction is staged under `/content` and rejects traversal paths and links before moving the discovered dataset root into the exact configured location.

In [ ]:
import shutil
import tarfile
import tempfile
from pathlib import Path, PurePosixPath

if not DRIVE_ARCHIVE_PATH.is_file():
    raise FileNotFoundError(
        f'Archive not found: {DRIVE_ARCHIVE_PATH}. Upload iam_words_colab.tar.gz to that Drive path '
        'or update DRIVE_ARCHIVE_PATH in the setup cell.'
    )
dataset_dir = PROJECT_DIR / 'data/raw/iam_words'
annotation_path = PROJECT_DIR / 'data/raw/iam_words/words.txt'
image_dir = PROJECT_DIR / 'data/raw/iam_words/words'

if annotation_path.is_file() and image_dir.is_dir():
    print('Dataset already extracted:', dataset_dir)
else:
    if dataset_dir.exists():
        raise FileExistsError(f'Incomplete dataset directory exists at {dataset_dir}; refusing to overwrite it.')
    dataset_dir.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(DRIVE_ARCHIVE_PATH, 'r:*') as archive:
        safe_members = []
        names = set()
        for member in archive.getmembers():
            member_path = PurePosixPath(member.name)
            if member_path.is_absolute() or '..' in member_path.parts:
                raise ValueError(f'Unsafe archive path: {member.name!r}')
            name = member_path.as_posix().removeprefix('./')
            if not name or name == '.':
                continue
            if name in names:
                raise ValueError(f'Duplicate archive member: {name}')
            names.add(name)
            if not (member.isdir() or member.isfile()):
                raise ValueError(f'Archive contains an unsupported link/file type: {member.name!r}')
            safe_members.append((member, PurePosixPath(name)))

        roots = set()
        for _, member_path in safe_members:
            if member_path.name != 'words.txt':
                continue
            root = member_path.parent
            words_prefix = (root / 'words').parts
            if any(path.parts[:len(words_prefix)] == words_prefix and path.suffix.lower() == '.png'
                   for _, path in safe_members):
                roots.add(root)
        if len(roots) != 1:
            raise ValueError(
                'Expected exactly one archive directory with words.txt and sibling words/*.png; '
                f'detected candidates: {sorted(map(str, roots))}'
            )
        archive_root = next(iter(roots))
        print('Detected dataset root in archive:', archive_root)

        with tempfile.TemporaryDirectory(prefix='iam_extract_', dir='/content') as temp_dir:
            staging = Path(temp_dir).resolve()
            for member, member_path in safe_members:
                target = (staging / Path(*member_path.parts)).resolve()
                if not target.is_relative_to(staging):
                    raise ValueError(f'Archive path escapes extraction directory: {member.name!r}')
                if member.isdir():
                    target.mkdir(parents=True, exist_ok=True)
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    source = archive.extractfile(member)
                    if source is None:
                        raise OSError(f'Could not read archive member: {member.name!r}')
                    with source, target.open('wb') as output:
                        shutil.copyfileobj(source, output)
            extracted = staging.joinpath(*archive_root.parts)
            if not (extracted / 'words.txt').is_file() or not (extracted / 'words').is_dir():
                raise FileNotFoundError('Discovered archive root lacks words.txt or words/.')
            shutil.move(str(extracted), str(dataset_dir))

if not annotation_path.is_file() or not image_dir.is_dir():
    raise FileNotFoundError(f'Expected IAM paths missing: {annotation_path} and {image_dir}')
record_count = sum(1 for line in annotation_path.open(encoding='utf-8') if line.strip() and not line.lstrip().startswith('#'))
image_count = sum(1 for _ in image_dir.rglob('*.png'))
if record_count == 0 or image_count == 0:
    raise ValueError(f'Dataset is empty: {record_count} annotation rows, {image_count} PNG images.')
print('Annotation:', annotation_path, '| records:', record_count)
print('Images:', image_dir, '| PNG count:', image_count)

## Dataset validation and processed metadata

Run the project's validation command against the extracted IAM files. Then load the existing splits and vocabulary with the project DataLoader implementation. If they are missing or invalid, preserve the invalid files with a timestamped suffix and regenerate them using the existing deterministic `--prepare-only --refresh-splits` command.

In [ ]:
import json
import subprocess
import sys
from datetime import datetime, timezone

from src.config import get_project_config
from src.train import build_data_loaders

config = get_project_config(PROJECT_DIR / 'configs/config.yaml')
assert config.annotation_file == annotation_path.resolve()
assert config.image_root == image_dir.resolve()
subprocess.check_call([
    sys.executable, '-m', 'src.dataset',
    '--annotation', str(config.annotation_file), '--image-root', str(config.image_root)
], cwd=PROJECT_DIR)

metadata_valid = False
if config.splits_path.is_file() and config.vocab_path.is_file():
    try:
        build_data_loaders(config, batch_size=2)
        metadata_valid = True
        print('Existing splits.json and vocabulary.json are valid.')
    except (ValueError, OSError, KeyError, TypeError, json.JSONDecodeError) as error:
        print('Existing processed metadata is invalid; it will be preserved and regenerated:', error)

if not metadata_valid:
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    config.processed_dir.mkdir(parents=True, exist_ok=True)
    for path in (config.splits_path, config.vocab_path):
        if path.exists():
            backup = path.with_name(f'{path.name}.invalid-{timestamp}')
            path.rename(backup)
            print('Preserved:', backup)
    subprocess.check_call([
        sys.executable, '-m', 'src.train', '--config', 'configs/config.yaml',
        '--prepare-only', '--refresh-splits'
    ], cwd=PROJECT_DIR)
    build_data_loaders(config, batch_size=2)
    print('Generated and validated splits.json and vocabulary.json.')
assert config.splits_path.is_file() and config.vocab_path.is_file()
print('Splits:', config.splits_path, '| vocabulary:', config.vocab_path)

## Route checkpoints and logs to persistent Drive

Training and extracted data stay in temporary `/content`; only checkpoints and logs persist in Drive. The repository training code writes to `checkpoints/` and `training_logs/`, so these local paths are linked to Drive without changing training code or configuration.

In [ ]:
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name in ('checkpoints', 'training_logs'):
    local_path = PROJECT_DIR / name
    drive_path = DRIVE_OUTPUT_DIR / name
    drive_path.mkdir(parents=True, exist_ok=True)
    if local_path.is_symlink():
        if local_path.resolve() != drive_path.resolve():
            raise RuntimeError(f'{local_path} points to an unexpected location.')
    elif local_path.exists():
        if not local_path.is_dir() or any(local_path.iterdir()):
            raise FileExistsError(f'{local_path} is not empty; refusing to replace local outputs.')
        local_path.rmdir()
        local_path.symlink_to(drive_path, target_is_directory=True)
    else:
        local_path.symlink_to(drive_path, target_is_directory=True)
    print(f'{local_path} -> {drive_path}')
assert (PROJECT_DIR / 'checkpoints').resolve().is_relative_to(DRIVE_OUTPUT_DIR.resolve())
assert (PROJECT_DIR / 'training_logs').resolve().is_relative_to(DRIVE_OUTPUT_DIR.resolve())

## One-batch GPU smoke test

This runs a forward pass and CTC loss on two training samples, checks model/collate lengths and finite loss, then enables the manual training cell. It does not start an epoch or write a checkpoint.

In [ ]:
import torch
from torch import nn

from src.model import CNNBiLSTMCTC
from src.train import _minimum_ctc_steps, build_data_loaders

SMOKE_TEST_PASSED = False
if not torch.cuda.is_available():
    raise RuntimeError('Smoke test requires CUDA. Select a GPU runtime before continuing.')
loaders, vocabulary, _ = build_data_loaders(config, batch_size=2)
device = torch.device('cuda')
model = CNNBiLSTMCTC(
    image_height=config.image_height, cnn_channels=config.cnn_channels,
    rnn_hidden_size=config.rnn_hidden_size, rnn_num_layers=config.rnn_num_layers,
    vocab_size=len(vocabulary), dropout=config.dropout
).to(device)
batch = next(iter(loaders['train']))
images, targets = batch['images'].to(device), batch['targets'].to(device)
logits = model(images)
input_lengths = model.compute_output_lengths(batch['image_widths'])
assert images.ndim == 4 and images.shape[1] == 1
assert logits.ndim == 3 and logits.shape[1] == images.shape[0] and logits.shape[2] == len(vocabulary)
assert torch.equal(input_lengths, batch['input_lengths'])
assert int(input_lengths.max()) == logits.shape[0]
for row, target_length in enumerate(batch['target_lengths'].tolist()):
    assert int(input_lengths[row]) >= _minimum_ctc_steps(targets[row, :target_length].cpu()), batch['word_ids'][row]
criterion = nn.CTCLoss(blank=vocabulary.index('<blank>'), zero_infinity=True)
loss = criterion(logits.log_softmax(dim=2), targets, input_lengths, batch['target_lengths'])
assert torch.isfinite(loss), loss
SMOKE_TEST_PASSED = True
print('GPU smoke test passed on', torch.cuda.get_device_name(0))
print('Images:', tuple(images.shape), '| logits:', tuple(logits.shape))
print('Input lengths:', input_lengths.tolist(), '| target lengths:', batch['target_lengths'].tolist())
print('Finite CTC loss:', float(loss.detach().item()))

## Full training — manual only

Run this cell only when ready. It uses the unchanged repository configuration and model. The training loop logs train/validation CTC loss and chooses the best checkpoint by validation performance; it does not use the test split for model selection.

In [ ]:
if not SMOKE_TEST_PASSED:
    raise RuntimeError('Run the GPU smoke-test cell successfully before full training.')
subprocess.check_call(
    [sys.executable, '-m', 'src.train', '--config', 'configs/config.yaml'],
    cwd=PROJECT_DIR,
)